
# Crop Recommendation AI
## Final EDA, Model Selection, Training and Evaluation

### Objective
This notebook develops an AI-powered crop recommendation system for agricultural extension agents.

The model uses:
- Soil nutrients (Nitrogen, Phosphorus, Potassium)
- Environmental conditions (Temperature, Humidity, Rainfall)
- Soil texture

The goal is to recommend suitable crops and provide evidence-based explanations.

Models evaluated:
1. Random Forest
2. XGBoost
3. Gradient Boosting

Evaluation metrics:
- Accuracy
- Macro F1 Score
- Weighted F1 Score
- Confusion Matrix

### Why these models?

Tree-based ensemble models are suitable for agricultural prediction because crop suitability depends on complex relationships between multiple factors.

For example:
- High rainfall alone does not determine rice suitability.
- Rainfall combined with humidity, soil texture and nutrient levels provides stronger prediction signals.



In [ ]:

# Import required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib



## 1. Load Dataset

The dataset contains soil and environmental conditions linked to recommended crop classes.


In [ ]:

df = pd.read_csv("original_dataset.csv")

df.head()


In [ ]:

# Check dataset structure

print("Dataset shape:", df.shape)

df.info()



## 2. Data Cleaning

The dataset contained exact duplicate observations.

Removing duplicates is important because:
- duplicated records can cause data leakage;
- the model may appear more accurate than it actually is;
- validation results become unrealistic.

Therefore, duplicates are removed before training.


In [ ]:

print("Duplicate rows:", df.duplicated().sum())

df_clean = df.drop_duplicates().reset_index(drop=True)

# Rename column to make it descriptive
df_clean = df_clean.rename(
    columns={"soil":"soil_texture"}
)

print("Clean dataset shape:", df_clean.shape)



## 3. Exploratory Data Analysis

Understanding the dataset helps identify:
- class balance;
- feature behaviour;
- possible relationships between soil/environment factors and crops.


In [ ]:

sns.countplot(
    data=df_clean,
    x="crop_label"
)

plt.xticks(rotation=45)
plt.title("Distribution of Crop Classes")
plt.show()


In [ ]:

numeric_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "ph",
    "rainfall"
]

plt.figure(figsize=(10,6))

sns.heatmap(
    df_clean[numeric_features].corr(),
    annot=True
)

plt.title("Feature Correlation")
plt.show()



## 4. Prepare Data For Modelling

The soil texture variable is categorical.

One-hot encoding converts soil categories into numerical values that machine learning models can understand.

Example:

loamy soil → [1,0,0,0]

sandy soil → [0,1,0,0]


In [ ]:

X = df_clean.drop("crop_label", axis=1)

y = df_clean["crop_label"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "soil_encoder",
            OneHotEncoder(handle_unknown="ignore"),
            ["soil_texture"]
        )
    ],
    remainder="passthrough"
)



# 5. Model Selection

Three ensemble models are compared.

## Random Forest
Uses multiple independent decision trees.

Advantages:
- robust;
- easy interpretation;
- good performance on small datasets.

## XGBoost
Builds trees sequentially and learns from previous errors.

Advantages:
- captures complex relationships;
- often performs strongly on structured/tabular data.

## Gradient Boosting
Another boosting approach that improves predictions sequentially.


In [ ]:

models = {

"Random Forest":
RandomForestClassifier(
    n_estimators=300,
    random_state=42
),

"XGBoost":
XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="mlogloss"
),

"Gradient Boosting":
GradientBoostingClassifier(
    random_state=42
)

}


results = {}


for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", model)
    ])

    pipeline.fit(
        X_train,
        y_train
    )


    predictions = pipeline.predict(
        X_test
    )


    results[name] = {

        "Accuracy":
        accuracy_score(
            y_test,
            predictions
        ),

        "Macro F1":
        f1_score(
            y_test,
            predictions,
            average="macro"
        ),

        "Weighted F1":
        f1_score(
            y_test,
            predictions,
            average="weighted"
        )

    }


pd.DataFrame(results).T



## Interpretation of Results

Accuracy alone is not enough.

Macro F1:
- treats every crop equally;
- important because every crop class should be predicted fairly.

Weighted F1:
- considers the number of samples in each crop class;
- useful when class sizes differ.

Because the dataset is balanced, Macro and Weighted F1 should be similar.

If all models achieve similar scores, model choice should consider:
- explainability;
- deployment;
- ability to justify recommendations.



In [ ]:

# Train final XGBoost model

final_model = Pipeline([
    ("preprocessing", preprocessor),
    (
        "model",
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=5,
            random_state=42,
            eval_metric="mlogloss"
        )
    )
])


final_model.fit(
    X_train,
    y_train
)


y_pred = final_model.predict(
    X_test
)


print(classification_report(
    y_test,
    y_pred
))



# 6. Confusion Matrix

The confusion matrix shows where the model correctly predicts crops and where errors occur.

A strong model should have:
- high values on the diagonal;
- few incorrect classifications outside the diagonal.


In [ ]:

cm = confusion_matrix(
    y_test,
    y_pred
)


plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=final_model.classes_,
    yticklabels=final_model.classes_
)

plt.xlabel("Predicted Crop")
plt.ylabel("Actual Crop")

plt.title("Confusion Matrix")

plt.show()



# 7. Save Model

The saved model will be used by Streamlit.

The application will:
- collect farm measurements from an agricultural agent;
- predict suitable crops;
- display the top recommendations.


In [ ]:

joblib.dump(
    final_model,
    "crop_prediction_model2.pkl"
)
